# FableForge-14B Fine-Tuning with Unsloth (SFT)

Fine-tune `Qwen/Qwen2.5-Coder-14B-Instruct` on multi-turn coding agent conversations.

**Runtime**: A100 40GB (Colab Pro) or multi-GPU | **Time**: ~2-3 hours | **VRAM**: ~24GB with 4-bit

⚠️ This model requires Colab Pro (A100) or a local GPU with 24GB+ VRAM.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y transformers && pip install transformers==4.46.3

from unsloth import FastLanguageModel
import torch

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct"
HF_REPO = "fableforge-ai/FableForge-14B"
MAX_SEQ_LENGTH = 4096  # 14B can handle longer sequences
LORA_R = 32  # Higher rank for larger model
BATCH_SIZE = 1  # Conservative for 14B on A100
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 1e-4  # Lower LR for larger model
NUM_EPOCHS = 3

# Load model with 4-bit QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f}GB")

In [ ]:
# Load SFT data
from datasets import Dataset
import json

TRAIN_PATH = "fableforge_sft_train.jsonl"
VAL_PATH = "fableforge_sft_val.jsonl"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

raw_train = load_jsonl(TRAIN_PATH)
raw_val = load_jsonl(VAL_PATH)

train_dataset = Dataset.from_list(raw_train)
val_dataset = Dataset.from_list(raw_val)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Sample keys: {train_dataset[0].keys()}")

In [ ]:
# Format conversations for training
def format_to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_to_text)
val_dataset = val_dataset.map(format_to_text)

# Verify formatting
print(train_dataset[0]["text"][:500])

In [ ]:
# Train with SFT
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        warmup_ratio=0.1,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
        save_steps=100,
        save_total_limit=3,
    ),
)

trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s")

In [ ]:
# Test inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are FableForge, an expert coding agent."},
    {"role": "user", "content": "Write a Python class that implements a rate limiter using the token bucket algorithm"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.3, top_p=0.9, use_cache=True)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Save LoRA adapters
model.save_pretrained("fableforge-14b-lora")
tokenizer.save_pretrained("fableforge-14b-lora")
print("LoRA adapters saved to ./fableforge-14b-lora")

In [ ]:
# Merge and save full model
model.save_pretrained_merged("fableforge-14b-merged", tokenizer)
print("Merged model saved to ./fableforge-14b-merged")

In [ ]:
# Push to HuggingFace
# from huggingface_hub import login
# login(token="your_token_here")
# model.push_to_hub_merged(HF_REPO, tokenizer)
# print(f"Model pushed to https://huggingface.co/{HF_REPO}")